# Gemini Tool Use: Reminder Assistant

Converted from the original Claude/Anthropic notebook. This version uses Google's `google-genai` SDK. The tool schemas are translated to Gemini function declarations; the actual multi-turn function-execution loop can be added in the next section.

In [7]:
# Install dependencies (run once if needed)
# %pip install -U google-genai python-dotenv

# Load environment variables and create Gemini client
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

# Set GEMINI_API_KEY in your .env file
client = genai.Client()
model = "gemini-2.5-flash"


In [8]:
# Helper functions
# Gemini conversation roles are: 'user' and 'model'.

def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "parts": [{"text": text}],
    }
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {
        "role": "model",
        "parts": [{"text": text}],
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    config = types.GenerateContentConfig(
        max_output_tokens=1000,
        temperature=temperature,
        stop_sequences=stop_sequences or [],
        system_instruction=system,
    )

    response = client.models.generate_content(
        model=model,
        contents=messages,
        config=config,
    )

    return response.text


In [ ]:
# Tools and Schemas

from datetime import datetime, timedelta


def get_current_datetime():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Gets the current date and time.",
    "parameters": {
        "type": "OBJECT",
        "properties": {},
        "required": []
    }
}

def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31, 30, 31, 30, 31, 31, 30, 31, 30, 31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    # Placeholder from the original notebook: this prints but does not
    # persist a reminder or send a notification yet.
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


# Gemini uses JSON Schema under `parameters`, rather than Claude's
# `input_schema` field.
add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a specified duration to a datetime string and returns the "
        "result in a detailed format. Supports seconds, minutes, hours, "
        "days, weeks, months, and years."
    ),
    "parameters": {
        "type": "OBJECT",
        "properties": {
            "datetime_str": {
                "type": "STRING",
                "description": "Input datetime string, formatted according to input_format.",
            },
            "duration": {
                "type": "NUMBER",
                "description": "Amount of time to add. Can be positive or negative.",
            },
            "unit": {
                "type": "STRING",
                "description": "One of: seconds, minutes, hours, days, weeks, months, years.",
            },
            "input_format": {
                "type": "STRING",
                "description": "Python strptime format, e.g. %Y-%m-%d.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": (
        "Creates a reminder for the specified content and timestamp. "
        "The current implementation is a placeholder and only prints the reminder."
    ),
    "parameters": {
        "type": "OBJECT",
        "properties": {
            "content": {
                "type": "STRING",
                "description": "Text describing what the user wants to be reminded about.",
            },
            "timestamp": {
                "type": "STRING",
                "description": "Scheduled date and time, preferably ISO 8601.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invokes multiple other tool calls. The application must execute each invocation.",
    "parameters": {
        "type": "OBJECT",
        "properties": {
            "invocations": {
                "type": "ARRAY",
                "description": "Tool calls to invoke.",
                "items": {
                    "type": "OBJECT",
                    "properties": {
                        "name": {"type": "STRING", "description": "Tool name."},
                        "arguments": {
                            "type": "STRING",
                            "description": "Arguments encoded as a JSON string.",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

# Register the function declarations with Gemini.
# These declarations make Gemini aware of the available functions;
# your Python application still has to execute any function calls.
reminder_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(**add_duration_to_datetime_schema),
        types.FunctionDeclaration(**set_reminder_schema),
        types.FunctionDeclaration(**batch_tool_schema),
        types.FunctionDeclaration(**get_current_datetime_schema)
    ]
)


In [10]:
from google.genai import types

messages = []

messages.append(
    {
        "role": "user",
        "parts": [
            {
                "text": "What is the exact time, formatted as HH:MM:SS?"
            }
        ]
    }
)

response = client.models.generate_content(
    model=model,
    contents=messages,
    config=types.GenerateContentConfig(
        tools=[get_current_datetime_schema]
    )
)

response

ValidationError: 3 validation errors for GenerateContentConfig
tools.name
  Extra inputs are not permitted [type=extra_forbidden, input_value='get_current_datetime', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden
tools.description
  Extra inputs are not permitted [type=extra_forbidden, input_value='Gets the current date and time.', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden
tools.parameters
  Extra inputs are not permitted [type=extra_forbidden, input_value={'type': 'OBJECT', 'prope...es': {}, 'required': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden

## Important differences and next step

- Claude's `input_schema` becomes Gemini's `parameters` in a function declaration.
- Gemini conversation history uses `role: "model"` for assistant turns and `parts` for message content.
- The `chat()` helper above is the text-generation baseline. It does **not** yet execute function calls. The next implementation step is to pass `reminder_tool` in `GenerateContentConfig(tools=[reminder_tool])`, inspect `response.function_calls`, execute the matching Python function, and send a `functionResponse` part back to Gemini.
- The original notebook does not yet define a `get_current_datetime` function/schema. We should add that when we reach the section that introduces the current-time tool.
- The original `set_reminder()` is only a print placeholder; it does not actually schedule a notification.
